In [2]:
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# File paths
# --------------------------------------------------

INPUT_FILE = Path("data/TDS.xlsx")
OUTPUT_FILE = Path("data/hindcon_master_dataset_sorted.xlsx")


# --------------------------------------------------
# Read Excel file
# --------------------------------------------------

df = pd.read_excel(INPUT_FILE)

print(f"Total rows before sorting: {len(df)}")


# --------------------------------------------------
# Check Product_Name column
# --------------------------------------------------

if "Product_Name" not in df.columns:
    raise ValueError("Product_Name column was not found in the Excel file.")


# --------------------------------------------------
# Clean Product_Name temporarily for sorting
# --------------------------------------------------

df["_Product_Name_Sort"] = (
    df["Product_Name"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)


# --------------------------------------------------
# Sort by Product_Name
# --------------------------------------------------

df = df.sort_values(
    by="_Product_Name_Sort",
    ascending=True,
    kind="stable"
)


# --------------------------------------------------
# Remove temporary column
# --------------------------------------------------

df = df.drop(columns=["_Product_Name_Sort"])


# --------------------------------------------------
# Reset row numbers
# --------------------------------------------------

df = df.reset_index(drop=True)


# --------------------------------------------------
# Save sorted Excel file
# --------------------------------------------------

df.to_excel(OUTPUT_FILE, index=False)


print(f"Sorted Excel file created successfully:")
print(OUTPUT_FILE)
print(f"Total rows after sorting: {len(df)}")

Total rows before sorting: 380
Sorted Excel file created successfully:
data\hindcon_master_dataset_sorted.xlsx
Total rows after sorting: 380


In [5]:
import pandas as pd
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed


# ============================================================
# FILE PATHS
# ============================================================

INPUT_FILE = Path("data/hindcon_master_dataset_sorted.xlsx")
OUTPUT_FILE = Path("data/tds_link_check_report_sort.xlsx")


# ============================================================
# SETTINGS
# ============================================================

TIMEOUT = 15
MAX_WORKERS = 10


# ============================================================
# CHECK ONE TDS LINK
# ============================================================

def check_tds_link(url):
    """
    Check whether a TDS URL is reachable.
    """

    if pd.isna(url) or str(url).strip() == "":
        return "NOT AVAILABLE", "No TDS link"

    url = str(url).strip()

    # Check URL format
    if not url.startswith(("http://", "https://")):
        return "INVALID", "Invalid URL format"

    try:
        # First try HEAD request
        response = requests.head(
            url,
            timeout=TIMEOUT,
            allow_redirects=True,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        status_code = response.status_code

        # Some servers don't support HEAD properly.
        # If HEAD fails, try GET.
        if status_code >= 400 or status_code == 405:
            response = requests.get(
                url,
                timeout=TIMEOUT,
                allow_redirects=True,
                headers={
                    "User-Agent": "Mozilla/5.0"
                },
                stream=True
            )

            status_code = response.status_code

        if 200 <= status_code < 400:
            return "WORKING", f"HTTP {status_code}"

        return "NOT WORKING", f"HTTP {status_code}"

    except requests.exceptions.Timeout:
        return "NOT WORKING", "Timeout"

    except requests.exceptions.SSLError:
        return "NOT WORKING", "SSL error"

    except requests.exceptions.ConnectionError:
        return "NOT WORKING", "Connection error"

    except requests.exceptions.RequestException as e:
        return "NOT WORKING", str(e)

    except Exception as e:
        return "NOT WORKING", str(e)


# ============================================================
# MAIN
# ============================================================

def main():

    print("Reading Excel file...")

    df = pd.read_excel(INPUT_FILE)

    if "TDS_Link" not in df.columns:
        raise ValueError("TDS_Link column was not found in the Excel file.")

    if "Product_Name" not in df.columns:
        raise ValueError("Product_Name column was not found in the Excel file.")

    print(f"Total rows: {len(df)}")
    print("Checking TDS links...")
    print()

    # --------------------------------------------------------
    # Prepare result columns
    # --------------------------------------------------------

    df["TDS_Check_Status"] = ""
    df["TDS_Check_Details"] = ""

    # --------------------------------------------------------
    # Check links in parallel
    # --------------------------------------------------------

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

        future_to_index = {}

        for index, url in df["TDS_Link"].items():

            future = executor.submit(
                check_tds_link,
                url
            )

            future_to_index[future] = index

        completed = 0

        for future in as_completed(future_to_index):

            index = future_to_index[future]

            status, details = future.result()

            df.at[index, "TDS_Check_Status"] = status
            df.at[index, "TDS_Check_Details"] = details

            completed += 1

            product_name = df.at[index, "Product_Name"]

            print(
                f"[{completed}/{len(df)}] "
                f"{product_name} → {status} ({details})"
            )

    # --------------------------------------------------------
    # Save complete report
    # --------------------------------------------------------

    df.to_excel(
        OUTPUT_FILE,
        index=False
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    working = (df["TDS_Check_Status"] == "WORKING").sum()
    not_working = (df["TDS_Check_Status"] == "NOT WORKING").sum()
    invalid = (df["TDS_Check_Status"] == "INVALID").sum()
    unavailable = (df["TDS_Check_Status"] == "NOT AVAILABLE").sum()

    print()
    print("=" * 60)
    print("TDS LINK CHECK COMPLETE")
    print("=" * 60)

    print(f"Total rows      : {len(df)}")
    print(f"Working links   : {working}")
    print(f"Not working     : {not_working}")
    print(f"Invalid links   : {invalid}")
    print(f"Not available   : {unavailable}")

    print()
    print(f"Report saved to:")
    print(OUTPUT_FILE)


if __name__ == "__main__":
    main()

Reading Excel file...
Total rows: 306
Checking TDS links...

[1/306] Crack Seal → WORKING (HTTP 200)
[2/306] FLOWPLAN 1500 → WORKING (HTTP 200)
[3/306] FLOWPLAN 1500 → WORKING (HTTP 200)
[4/306] FLOWPLAN 1250 → WORKING (HTTP 200)
[5/306] FLOWPLAN 1250 → WORKING (HTTP 200)
[6/306] E3-G-IN → WORKING (HTTP 200)
[7/306] FLOWPLAN 2500 → WORKING (HTTP 200)
[8/306] FLOWPLAN 2500 → WORKING (HTTP 200)
[9/306] COLLOIDAL SILICA (CS40) → WORKING (HTTP 200)
[10/306] COLLOIDAL SILICA (CS30) → WORKING (HTTP 200)
[11/306] HBD500 → WORKING (HTTP 200)
[12/306] Hind Acrylic Wall Putty → WORKING (HTTP 200)
[13/306] Hind Actcolor → WORKING (HTTP 200)
[14/306] Hind Anti Rust → WORKING (HTTP 200)
[15/306] Hind Anti Rust EZ → WORKING (HTTP 200)
[16/306] Hind Anti Rust → WORKING (HTTP 200)
[17/306] Hind Bitkote → WORKING (HTTP 200)
[18/306] Hind Anti Air → WORKING (HTTP 200)
[19/306] Hind Anchorlok → WORKING (HTTP 200)
[20/306] Hind Basebond → WORKING (HTTP 200)
[21/306] Hind Bitkote (CT) → WORKING (HTTP 200)
